In [1]:
"""
Summary:
    This script demonstrates a Multi-Head Attention block with support for KV-Caching.
    KV-Caching allows us to reuse previously computed key (K) and value (V) tensors during
    autoregressive generation, avoiding redundant computation. This is useful during inference,
    while during training the full sequence is processed and caching is typically not used.

    The code includes detailed explanations on tensor shape transformations:
      Input: (batch, seq_len, d_model)
      After linear projection and reshaping: (batch, seq_len, d_model) --> (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
      During caching, new keys/values of shape (batch, h, new_seq_len, d_k) are concatenated with cached ones 
      of shape (batch, h, seq_len_cached, d_k) to form (batch, h, seq_len_total, d_k).

Example:
    >>> mha = MultiHeadAttentionBlock(d_model=512, h=8, dropout=0.1)
    >>> cache = {"cached_k": None, "cached_v": None}
    >>> output, cache = mha(q, k, v, mask, cache)
"""

import math
import logging
from typing import Optional, Dict, Tuple

import torch
import torch.nn as nn

# Configuration constants
CONFIG = {
    "DROPOUT_PROB": 0.1,         # Dropout probability for attention scores
    "LOGGING_LEVEL": logging.INFO,
}

logging.basicConfig(level=CONFIG["LOGGING_LEVEL"])


class MultiHeadAttentionBlock(nn.Module):
    """
    Multi-Head Attention block with support for KV-Caching.

    KV-Caching:
      - Stores previously computed key (K) and value (V) tensors.
      - During autoregressive inference, new keys/values (for new tokens) are concatenated with cached keys/values.

    Shape Transformations:
      1. Input q, k, v: (batch, seq_len, d_model)
      2. After linear projection: (batch, seq_len, d_model)
      3. Reshape to separate heads:
            (batch, seq_len, d_model) --> (batch, seq_len, h, d_k)
      4. Transpose to bring head dimension upfront:
            (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
      5. For KV-Caching (if used):
            New keys/values: (batch, h, new_seq_len, d_k)
            Cached keys/values: (batch, h, seq_len_cached, d_k)
            Concatenated result: (batch, h, seq_len_total, d_k)
    """

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        """
        Initialize the MultiHeadAttentionBlock.

        Args:
            d_model (int): Dimension of the embedding vectors.
            h (int): Number of attention heads.
            dropout (float): Dropout probability.
        """
        super().__init__()
        self.d_model: int = d_model  # Embedding dimension
        self.h: int = h              # Number of heads
        # Ensure d_model is divisible by h
        assert d_model % h == 0, "d_model must be divisible by h"
        self.d_k: int = d_model // h  # Dimension per head

        # Define linear layers for computing query, key, value, and output
        self.w_q: nn.Linear = nn.Linear(d_model, d_model, bias=False)
        self.w_k: nn.Linear = nn.Linear(d_model, d_model, bias=False)
        self.w_v: nn.Linear = nn.Linear(d_model, d_model, bias=False)
        self.w_o: nn.Linear = nn.Linear(d_model, d_model, bias=False)
        self.dropout: nn.Dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        d_k = query.shape[-1]
        # Just apply the formula from the paper
        # (batch, h, seq_len, d_k) --> (batch, h, seq_len, seq_len)
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)
        if mask is not None:
            # Write a very low value (indicating -inf) to the positions where mask == 0
            attention_scores.masked_fill_(mask == 0, -1e9)
        attention_scores = attention_scores.softmax(dim=-1) # (batch, h, seq_len, seq_len) # Apply softmax
        if dropout is not None:
            attention_scores = dropout(attention_scores)
        # (batch, h, seq_len, seq_len) --> (batch, h, seq_len, d_k)
        # return attention scores which can be used for visualization
        return (attention_scores @ value), attention_scores

        return output, attention_scores

    def forward(self,
                q: torch.Tensor,
                k: torch.Tensor,
                v: torch.Tensor,
                mask: Optional[torch.Tensor],
                cache: Optional[Dict[str, torch.Tensor]] = None) -> Tuple[torch.Tensor, Dict[str, torch.Tensor]]:
        """
        Forward pass for the Multi-Head Attention block with optional KV-Caching.

        Args:
            q (Tensor): Query tensor of shape (batch, seq_len, d_model)
            k (Tensor): Key tensor of shape (batch, seq_len, d_model)
            v (Tensor): Value tensor of shape (batch, seq_len, d_model)
            mask (Optional[Tensor]): Attention mask of shape (batch, 1, seq_len, seq_len_total)
            cache (Optional[Dict[str, Tensor]]): Cache dict with:
                - "cached_k": previously computed keys, shape (batch, h, seq_len_cached, d_k)
                - "cached_v": previously computed values, shape (batch, h, seq_len_cached, d_k)
                If provided, new keys/values are concatenated to these cached tensors.

        Returns:
            Tuple[Tensor, Dict[str, Tensor]]:
                - output: Output tensor of shape (batch, seq_len, d_model)
                - cache: Updated cache dict with concatenated keys and values.
        """
        try:
            # Step 1: Linear projection of inputs.
            # Input: (batch, seq_len, d_model) --> Output: (batch, seq_len, d_model)
            query: torch.Tensor = self.w_q(q)
            key: torch.Tensor = self.w_k(k)
            value: torch.Tensor = self.w_v(v)

            # Step 2: Reshape and transpose to separate heads.
            # Reshape: (batch, seq_len, d_model) --> (batch, seq_len, h, d_k)
            # Transpose: (batch, seq_len, h, d_k) --> (batch, h, seq_len, d_k)
            query = query.view(query.size(0), query.size(1), self.h, self.d_k).transpose(1, 2)  # (batch, h, seq_len, d_k)
            key = key.view(key.size(0), key.size(1), self.h, self.d_k).transpose(1, 2)        # (batch, h, seq_len, d_k)
            value = value.view(value.size(0), value.size(1), self.h, self.d_k).transpose(1, 2)  # (batch, h, seq_len, d_k)

            # Step 3: Apply KV-Caching if available.
            if cache is not None and "cached_k" in cache and "cached_v" in cache:
                if cache["cached_k"] is not None and cache["cached_v"] is not None:
                    # Concatenate new keys/values with cached ones along the sequence dimension (dim=2).
                    # Cached shape: (batch, h, seq_len_cached, d_k)
                    # New keys shape: (batch, h, new_seq_len, d_k)
                    # After concatenation: (batch, h, seq_len_total, d_k)
                    key = torch.cat([cache["cached_k"], key], dim=2)
                    value = torch.cat([cache["cached_v"], value], dim=2)
                    logging.info(f"KV caching: concatenated key shape {key.shape}, value shape {value.shape}.")
                # Update the cache with the concatenated keys and values.
                cache["cached_k"] = key
                cache["cached_v"] = value
            else:
                # If no cache provided, initialize cache with current keys and values.
                cache = {"cached_k": key, "cached_v": value}

            # Step 4: Compute attention.
            # query shape: (batch, h, seq_len, d_k)
            # key shape: (batch, h, seq_len_total, d_k) if caching is used, otherwise (batch, h, seq_len, d_k)
            # value shape: (batch, h, seq_len_total, d_k)
            x, self.attention_scores = MultiHeadAttentionBlock.attention(query, key, value, mask, self.dropout)
            # x shape: (batch, h, seq_len, d_k)

            # Step 5: Reassemble heads.
            # Transpose: (batch, h, seq_len, d_k) --> (batch, seq_len, h, d_k)
            # Reshape: (batch, seq_len, h, d_k) --> (batch, seq_len, d_model)
            x = x.transpose(1, 2).contiguous().view(x.size(0), x.size(2), self.h * self.d_k)  # (batch, seq_len, d_model)

            # Step 6: Final linear projection.
            # Input: (batch, seq_len, d_model) --> Output: (batch, seq_len, d_model)
            output: torch.Tensor = self.w_o(x)

            return output, cache

        except Exception as e:
            logging.error("Error during forward pass: " + str(e))
            raise e


if __name__ == "__main__":
    # Example test for Multi-Head Attention with KV-Caching

    # Test configuration
    batch_size: int = 2
    seq_len: int = 5      # Initial sequence length for training mode
    d_model: int = 512
    h: int = 8

    # Generate random input tensors for q, k, v with shape: (batch, seq_len, d_model)
    q: torch.Tensor = torch.randn(batch_size, seq_len, d_model)
    k: torch.Tensor = torch.randn(batch_size, seq_len, d_model)
    v: torch.Tensor = torch.randn(batch_size, seq_len, d_model)

    # Dummy mask: (batch, 1, seq_len, seq_len)
    mask: torch.Tensor = torch.ones(batch_size, 1, seq_len, seq_len)

    # Initialize cache as None (no caching during training)
    cache: Dict[str, Optional[torch.Tensor]] = {"cached_k": None, "cached_v": None}

    # Initialize the Multi-Head Attention block
    mha = MultiHeadAttentionBlock(d_model=d_model, h=h, dropout=CONFIG["DROPOUT_PROB"])

    # First pass (training mode: process full sequence without caching)
    output, cache = mha(q, k, v, mask, cache)
    logging.info(f"Output shape after first pass: {output.shape}")  # Expected: (batch, seq_len, d_model)

    # Simulate inference: processing one new token at a time with KV-Caching
    seq_len_new: int = 1  # New token sequence length
    q_new: torch.Tensor = torch.randn(batch_size, seq_len_new, d_model)
    k_new: torch.Tensor = torch.randn(batch_size, seq_len_new, d_model)
    v_new: torch.Tensor = torch.randn(batch_size, seq_len_new, d_model)
    # Create a mask that covers both cached tokens and the new token.
    # Mask shape: (batch, 1, new_seq_len, seq_len_total) where seq_len_total = cached_seq_len + new_seq_len
    mask_new: torch.Tensor = torch.ones(batch_size, 1, seq_len_new, cache["cached_k"].size(2) + seq_len_new)

    output, cache = mha(q_new, k_new, v_new, mask_new, cache)
    logging.info(f"Output shape after processing new token: {output.shape}")  # Expected: (batch, seq_len_new, d_model)

    print("Script executed successfully.")


INFO:root:Output shape after first pass: torch.Size([2, 5, 512])
INFO:root:KV caching: concatenated key shape torch.Size([2, 8, 6, 64]), value shape torch.Size([2, 8, 6, 64]).
INFO:root:Output shape after processing new token: torch.Size([2, 1, 512])


Script executed successfully.
